# Hidden Pfaffian ViT ground-state search

Small spinful Hubbard example using the hidden Pfaffian plus ViT coefficient network.

In [5]:
from itertools import combinations

import jax

jax.config.update("jax_enable_x64", True)

import jax.numpy as jnp
import jVMC_exp
import jVMC_exp.nets as nets
import jVMC_exp.operator.discrete as op
import jVMC_exp.sampler as sampler
from jVMC_exp.stats import SampledObs
from jVMC_exp.symmetry.lattice_symetries import square_translation_symmetry
from jVMC_exp.vqs import NQS

In [6]:
Lx, Ly = 2, 2
L = Lx * Ly
t = 1.0
U = 4.0
mu = U / 2.0

def site(x, y):
    return y * Lx + x

bonds = set()
for y in range(Ly):
    for x in range(Lx):
        i = site(x, y)
        bonds.add(tuple(sorted((i, site((x + 1) % Lx, y)))))
        bonds.add(tuple(sorted((i, site(x, (y + 1) % Ly)))))

hamiltonian = 0
for i, j in sorted(bonds):
    for spin in ("up", "down"):
        hamiltonian += -t * (op.Creation(i, spin) * op.Annihilation(j, spin))
        hamiltonian += -t * (op.Creation(j, spin) * op.Annihilation(i, spin))
for i in range(L):
    hamiltonian += U * op.Number(i, "up") * op.Number(i, "down")
    hamiltonian += -mu * (op.Number(i, "up") + op.Number(i, "down"))

In [7]:
def spinful_sector_basis(L, n_up, n_down, sample_shape):
    states = []
    for up_sites in combinations(range(L), n_up):
        up_sites = set(up_sites)
        for down_sites in combinations(range(L), n_down):
            down_sites = set(down_sites)
            state = []
            for site_idx in range(L):
                local_state = int(site_idx in up_sites) + 2 * int(site_idx in down_sites)
                state.append(local_state)
            states.append(state)
    return jnp.asarray(states, dtype=jnp.int32).reshape((-1, *sample_shape))


class FixedSpinfulSectorSampler(sampler.AbstractSampler):
    def __init__(self, psi, L, n_up, n_down):
        super().__init__(psi)
        self.basis = spinful_sector_basis(L, n_up, n_down, psi.sampleShape)

    def __call__(self, observable, **obs_kwargs):
        raw_data = observable.get_O_loc(self.samples, self.psi, logPsiS=self.logPsi, **obs_kwargs)
        return SampledObs(raw_data, self.weights)

    def sample(self, numSamples=None):
        del numSamples
        log_psi = self.psi(self.basis)
        shift = jnp.max(jnp.real(log_psi))
        weights = jnp.exp((jnp.real(log_psi) - shift) / 0.5)
        weights = weights / jnp.sum(weights)
        self._samples = self.basis
        self._logPsi = log_psi
        self._weights = weights
        return self._samples, self._logPsi, self._weights


symmetry = square_translation_symmetry(Lx, Ly, "spinful_fermion")
net = nets.HiddenPfaffianViT(
    Lx=Lx,
    Ly=Ly,
    symmetry=symmetry,
    patch_size=1,
    layers=2,
    embed_dim=16,
    heads=2,
    particles_per_spin=2,
    param_dtype=jnp.float64,
    compute_dtype=jnp.float64,
)
psi = NQS(net, (Ly, Lx), batchSize=64, seed=1234)
exact_sampler = FixedSpinfulSectorSampler(psi, L, n_up=2, n_down=2)
print("Number of parameters:", psi.numParameters)

Number of parameters: 5904


In [8]:
loss_function = jVMC_exp.objective_function.Observable(hamiltonian)
optimizer = jVMC_exp.optimizer.Adam(exact_sampler, psi, learning_rate=2.5e-3)

target_variance = 1e-5
block_steps = 250
max_blocks = 12

for block in range(max_blocks):
    optimizer.ground_state_search(block_steps, loss_function)
    exact_sampler.sample()
    final = exact_sampler(hamiltonian)
    variance = float(jnp.real(final.var[0]))
    energy = complex(final.mean[0])
    print(f"block {block + 1:02d}: E={energy.real:.12f}, Var={variance:.3e}")
    if variance < target_variance:
        break

print("Final local-energy estimate:", final)

100%|██████████| 250/250 [00:08<00:00, 28.73it/s, E=-1.0101e+01+1.4908e-19j ± 3.8202e-02 (Var = 1.0786e-02)]

Recorded timings:
    - sampling: 0.610257s
    - compute objective function and gradient: 4.090526s
    - solve: 0.680656s


block 01: E=-10.101180929727, Var=1.069e-02


100%|██████████| 250/250 [00:04<00:00, 51.90it/s, E=-1.0102e+01+2.3039e-19j ± 2.1680e-02 (Var = 3.4757e-03)]

Recorded timings:
    - sampling: 0.186085s
    - compute objective function and gradient: 1.285497s
    - solve: 0.370161s


...

100%|██████████| 250/250 [00:04<00:00, 60.64it/s, E=-1.0103e+01+1.6517e-19j ± 1.6591e-03 (Var = 2.0361e-05)]

Recorded timings:
    - sampling: 0.164574s
    - compute objective function and gradient: 1.176823s
    - solve: 0.295119s


block 10: E=-10.102745890063, Var=2.029e-05


100%|██████████| 250/250 [00:04<00:00, 53.09it/s, E=-1.0103e+01-7.1151e-20j ± 1.0793e-03 (Var = 8.6162e-06)]

Recorded timings:
    - sampling: 0.184612s
    - compute objective function and gradient: 1.284661s
    - solve: 0.354401s
block 11: E=-10.102747394731, Var=8.585e-06
Final local-energy estimate: -1.0103e+01-1.1011e-20j ± 1.0773e-03 (Var = 8.5850e-06)
